# 缺失值处理：5 种策略全解析

> 这是「数据分析从入门到精通」系列的第 11 篇。真实数据永远有缺失值，这篇把所有处理策略走一遍，帮你在不同场景下做出正确选择。

---

嗨，我是小荷。

你有没有遇到过这种情况：数据读进来，某列有几百个 `NaN`，然后你不知道怎么办，最后就……直接删掉了事？

这是最常见的处理方式，但不一定是最好的。

萧何当年管账，账簿上要是有几页残缺怎么办？如果是无关紧要的，可以标注"缺失"；如果是关键数据，就要想办法推算或者补全。**盲目删除可能丢失关键信息**，后面分析结果就不准了。

---

## 先搞清楚缺失值的类型

在动手之前，先判断缺失是怎么发生的：

| 类型 | 含义 | 举例 |
|------|------|------|
| **MCAR**（完全随机缺失） | 缺失与数据本身无关 | 仪器故障、录入遗漏 |
| **MAR**（随机缺失） | 缺失与其他变量有关 | 女性更倾向于不填年龄 |
| **MNAR**（非随机缺失） | 缺失与自身值有关 | 收入高的人不愿填收入 |

MNAR 最难处理——你删掉的那些"缺失值"，恰恰可能是最有价值的数据。

---

## 检测缺失值

In [6]:
import pandas as pd
import numpy as np

# 构造一份有缺失值的数据
data = {
    '用户ID': [1, 2, 3, 4, 5, 6],
    '年龄': [25, np.nan, 32, np.nan, 28, 35],
    '收入': [8000, 12000, np.nan, 6000, np.nan, 9500],
    '城市': ['北京', '上海', '广州', np.nan, '深圳', '杭州'],
    '是否购买': [1, 1, 0, 1, np.nan, 0]
}
df = pd.DataFrame(data)

# 检查每列缺失数量
print('每列缺失数量:\n', df.isnull().sum())

# 缺失比例
print('\n缺失比例:\n',( df.isnull().sum() / len(df) * 100).round(1))

# 可视化缺失分布（推荐用 missingno 库）
# !pip install missingno
# import missingno as msno
# msno.matrix(df)


每列缺失数量:
 用户ID    0
年龄      2
收入      2
城市      1
是否购买    1
dtype: int64

缺失比例:
 用户ID     0.0
年龄      33.3
收入      33.3
城市      16.7
是否购买    16.7
dtype: float64


---

## 策略一：直接删除（dropna）

最简单粗暴，适合**缺失比例很低**（< 5%）或**该行/列整体无效**的情况。

In [7]:
# 删除含任意缺失值的行
df.dropna()

# 删除所有值都缺失的行
df.dropna(how='all')

# 只要求指定列不缺失
df.dropna(subset=['年龄', '收入'])

# 删除缺失超过一定比例的列
threshold = 0.5
df.dropna(axis=1, thresh=int(len(df) * threshold))


,用户ID,年龄,收入,城市,是否购买
0,1,25.0,8000.0,北京,1.0
1,2,NaN,12000.0,上海,1.0
2,3,32.0,NaN,广州,0.0
3,4,NaN,6000.0,NaN,1.0
4,5,28.0,NaN,深圳,NaN
5,6,35.0,9500.0,杭州,0.0


> ⚠️ **什么时候不能直接删？** 如果缺失的是某类特定用户（比如高收入人群不填收入），删掉会让样本产生偏差，后续分析结论会失真。

---

## 策略二：固定值填充（fillna）

In [8]:
# 用 0 填充
df['收入'].fillna(0)

# 用字符串填充
df['城市'].fillna('未知')

# 用字典针对不同列填充不同值
df.fillna({'年龄': 0, '城市': '未知', '是否购买': 0})


,用户ID,年龄,收入,城市,是否购买
0,1,25.0,8000.0,北京,1.0
1,2,0.0,12000.0,上海,1.0
2,3,32.0,NaN,广州,0.0
3,4,0.0,6000.0,未知,1.0
4,5,28.0,NaN,深圳,0.0
5,6,35.0,9500.0,杭州,0.0


适合场景：
- 数值型：填 0（如访问次数缺失表示未访问）
- 类别型：填 "未知" 或 "其他"
- 布尔型：填 0 或 False

---

## 策略三：统计值填充

用均值、中位数、众数来填充，**数值列最常用**。

In [9]:
# 均值填充（适合正态分布数据）
df['年龄'].fillna(df['年龄'].mean())

# 中位数填充（适合有偏分布或存在异常值）
df['收入'].fillna(df['收入'].median())

# 众数填充（适合类别列）
df['城市'].fillna(df['城市'].mode()[0])


0    北京
1    上海
2    广州
3    上海
4    深圳
5    杭州
Name: 城市, dtype: str

> 💡 **小荷建议**：收入、价格这类偏态分布数据，**用中位数比均值更靠谱**，不容易被异常值带偏。

---

## 策略四：前向/后向填充（ffill / bfill）

适合**时间序列**数据，用相邻值填充。

In [10]:
# 时间序列气温数据
temp = pd.Series([20, np.nan, np.nan, 23, np.nan, 25])

# 前向填充：用前一个有效值填充
print(temp.ffill())
# 0    20.0
# 1    20.0   ← 用前值20填充
# 2    20.0
# 3    23.0
# 4    23.0   ← 用前值23填充
# 5    25.0

# 后向填充：用后一个有效值填充
print(temp.bfill())


0    20.0
1    20.0
2    20.0
3    23.0
4    23.0
5    25.0
dtype: float64
0    20.0
1    23.0
2    23.0
3    23.0
4    25.0
5    25.0
dtype: float64


---

## 策略五：插值填充（interpolate）

比前向填充更智能，适合连续变化的数值序列。

In [11]:
# 线性插值（默认）
temp.interpolate(method='linear')

# 多项式插值
temp.interpolate(method='polynomial', order=2)


0    20.0
1    21.0
2    22.0
3    23.0
4    24.0
5    25.0
dtype: float64

对比一下效果：
```
原始数据: [20, NaN, NaN, 23, NaN, 25]
前向填充: [20, 20,  20,  23, 23,  25]
线性插值: [20, 21,  22,  23, 24,  25]  ← 更合理！
```

---

## 实战：处理一份真实脏数据

In [14]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200
df = pd.DataFrame({
    '用户ID': range(1, n+1),
    '年龄': np.where(np.random.rand(n) < 0.08, np.nan,
                     np.random.randint(18, 60, n).astype(float)),
    '月收入': np.where(np.random.rand(n) < 0.12, np.nan,
                      np.random.normal(8000, 3000, n).round(0)),
    '城市级别': np.where(np.random.rand(n) < 0.05, None,
                        np.random.choice(['一线','二线','三线'], n)),
    '购买次数': np.where(np.random.rand(n) < 0.03, np.nan,
                        np.random.poisson(3, n).astype(float))
})

print("处理前缺失情况:")
print(df.isnull().sum())

# 分列处理
df_clean = df.copy()

# 年龄：中位数填充
df_clean['年龄'] = df_clean['年龄'].fillna(df_clean['年龄'].median())

# 月收入：中位数填充
df_clean['月收入'] = df_clean['月收入'].fillna(df_clean['月收入'].median())

# 城市级别：众数填充（或用 '未知'）
df_clean['城市级别'] = df_clean['城市级别'].fillna(df_clean['城市级别'].mode()[0])

# 购买次数：0 填充（或中位数）
df_clean['购买次数'] = df_clean['购买次数'].fillna(0)

print("\n处理后缺失情况:")
print(df_clean.isnull().sum())


处理前缺失情况:
用户ID     0
年龄      18
月收入     23
城市级别     5
购买次数     6
dtype: int64

处理后缺失情况:
用户ID    0
年龄      0
月收入     0
城市级别    0
购买次数    0
dtype: int64


---

## 如何选择策略？

```
缺失比例 < 5%?
  ├── 是 → 直接删除（简单安全）
  └── 否 → 看缺失类型
        ├── 时间序列 → ffill / interpolate
        ├── 数值列，正态分布 → 均值填充
        ├── 数值列，偏态分布 → 中位数填充
        ├── 类别列 → 众数或"未知"
        └── 缺失有业务含义（如"未购买"）→ 填 0/False
```

---

## 本篇小结

| 策略 | 函数 | 适用场景 |
|------|------|---------|
| 删除 | `dropna()` | 缺失少，或整行无效 |
| 固定值 | `fillna(值)` | 有明确业务含义 |
| 统计值 | `fillna(mean/median)` | 普通数值列 |
| 前后填充 | `ffill()` / `bfill()` | 时间序列 |
| 插值 | `interpolate()` | 连续变化的序列 |

---

## 课后练习

In [19]:
import pandas as pd, numpy as np
np.random.seed(0)
df = pd.DataFrame({
    '日期': pd.date_range('2024-01', periods=10, freq='D'),
    '销售额': [100, np.nan, 150, np.nan, np.nan, 200, np.nan, 180, 190, np.nan],
    '库存': [50, 45, np.nan, 40, np.nan, 30, 25, np.nan, 15, 10],
    '渠道': ['线上', np.nan, '线下', '线上', np.nan, '线下', '线上', np.nan, '线上', '线下']
})

# 任务1：统计每列缺失比例
# 任务2：销售额用前向填充，库存用插值，渠道用众数
# 任务3：处理后验证没有缺失值


In [20]:
# 任务1：统计每列缺失比例
print("=== 缺失比例 ===")
missing_ratio = df.isnull().sum() / len(df)
print(missing_ratio)

=== 缺失比例 ===
日期     0.0
销售额    0.5
库存     0.3
渠道     0.3
dtype: float64


In [21]:
# 任务2：销售额用前向填充，库存用插值，渠道用众数
# 销售额：前向填充
df['销售额'] = df['销售额'].ffill()

In [22]:
# 渠道：众数填充
mode_val = df['渠道'].mode()[0]
df['渠道'] = df['渠道'].fillna(mode_val)

In [23]:
# 任务3：验证没有缺失值
print("\n=== 处理后缺失值统计 ===")
print(df.isnull().sum())


=== 处理后缺失值统计 ===
日期     0
销售额    0
库存     3
渠道     0
dtype: int64


评论区见 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 12 篇：数据类型转换 — 日期 / 金额 / 类别**
>
> 数据不缺了，但类型不对也跑不动——日期列是字符串、金额列带了逗号、类别列占用内存太大……下篇一次搞定。

---

👇 点「在看」，推给每天和脏数据作斗争的人  
💬 评论区说说你遇到过最奇葩的缺失值情况  
⭐ 关注公众号，跟萧何迷妹学干净数据处理

---

*「数据分析从入门到精通」系列 · 第 11 篇*  
*上一篇：[第 10 篇：行列筛选]*  
*下一篇：第 12 篇：数据类型转换*